In [ ]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython import get_ipython
from pprint import pformat

formatter = get_ipython().display_formatter.formatters["text/plain"]
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))


In [ ]:
import numpy as np
import torch


def seed_everything(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


seed_everything(42)

In [ ]:
editing_methods = [
    "ft_l", "ft_m",
    "rome",
    "memit",
    "grace",
    "wise_retrieve", "wise_merge",
    "alpha_edit",
    "mnemo",
][:]
model_name = ["qwen2.5-7b"][0]

dataset_variant = ["cloze", "qna"][0]
# eval_milestones = sorted([1, 5, 10, 50, 100, 500, 1000, 1500, 2000])
eval_milestones = sorted([1, 5, 10, 25, 50, 75, 100])
eval_batch_size = 8

# n_runs = 5
n_runs = 3
base_seed = 0

model_generation = {
    "max_new_tokens": 32,
    # "do_sample": True,
    # "temperature": 0.7,
    # "top_p": 0.8,
    # "top_k": 20,
}

llm_judge = {
    # "base_url": "http://localhost:8080/v1",
    # "api_key": "local",
    "judge_model": "openai/gpt-oss-20b",
    "judge_extra_config": {
        "temperature": 0.1,
        "seed": base_seed,
        "extra_body": {
            "reasoning_effort": "low",
        },
    },
    "max_concurrency": 5,
    "max_retries": 5,
    # "init_retry_delay": 1.0,
    # "max_retry_delay": 30.0,
    # "repeat_times": 5,
    "repeat_times": 3,
}


In [ ]:
from collections.abc import Callable
from itertools import chain
from random import Random
from statistics import mean

from datasets import load_from_disk

from src.editors import EditRequest
from src.evaluate import EvalRecord, EvalResult, MetricResult


cfact = load_from_disk("../data/datasets/cfact")
mquake = load_from_disk("../data/datasets/mquake")


def build_inputs(
    dataset,
    build_pair: Callable[[object], tuple[EditRequest, EvalResult]],
    seed: int,
) -> tuple[list[EditRequest], list[EvalResult]]:

    n_records = eval_milestones[-1]

    if n_records <= 0:
        raise ValueError("n_records must be > 0")
    if n_records > len(dataset):
        raise ValueError(f"Requested {n_records} records, but only {len(dataset)} records are available")

    selected = list(dataset)
    Random(seed).shuffle(selected)
    selected = selected[:n_records]

    pairs = [build_pair(record) for record in selected]
    edit_requests, eval_results = zip(*pairs)
    return list(edit_requests), list(eval_results)


def aggregate_eval_results(results: list[EvalResult]) -> EvalResult:
    if not results:
        return EvalResult()

    def _aggregate_metric(metric_name: str) -> MetricResult | None:
        metric_results = [getattr(result, metric_name) for result in results]
        metric_results = [metric for metric in metric_results if metric is not None]
        if not metric_results:
            return None

        return MetricResult(
            score=mean(metric.score for metric in metric_results),
            details=list(chain.from_iterable(metric.details for metric in metric_results)),
        )

    return EvalResult(
        success=_aggregate_metric("success"),
        locality=_aggregate_metric("locality"),
        portability=_aggregate_metric("portability"),
    )


def format_step_log(step: int, total_steps: int, result: EvalResult) -> str:
    parts: list[str] = []

    if result.success is not None:
        parts.append(f"success: {result.success.score:.2f} (n={len(result.success.details)})")
    if result.locality is not None:
        parts.append(f"locality: {result.locality.score:.2f} (n={len(result.locality.details)})")
    if result.portability is not None:
        parts.append(f"portability: {result.portability.score:.2f} (n={len(result.portability.details)})")

    metrics_text = " --- ".join(parts) if parts else "no metrics"
    return f"[{step}/{total_steps}] {metrics_text}"


In [ ]:
from dataclasses import asdict
from time import perf_counter

from tqdm.notebook import trange

from src.editors import Editor, EditRequest, load_editor
from src.evaluate import EvalResult, evaluate, save_benchmark_results


GLOBAL_MODEL = None
GLOBAL_TOKENIZER = None


async def run_method_benchmark(
    editor: Editor,
    edit_requests: list[EditRequest],
    eval_results: list[EvalResult],
    editing_method: str,
    max_step: int,
    run_index: int,
) -> list[dict]:

    print(
        f"[method-start] "
        f"run={run_index}/{n_runs} "
        f"method={editing_method}"
    )

    run_start = perf_counter()
    total_edit_time = 0.0
    milestones = set(eval_milestones)
    milestone_results: list[dict] = []

    for step in trange(1, max_step + 1, desc=f"Benchmark [{editing_method}]"):
        edit_start = perf_counter()
        editor.edit(edit_requests[step - 1])
        edit_time = perf_counter() - edit_start
        total_edit_time += edit_time

        if step not in milestones:
            continue

        step_eval_results = await evaluate(
            editor.model,
            editor.tokenizer,
            eval_results[:step],
            batch_size=eval_batch_size,
            model_generation=model_generation,
            llm_judge=llm_judge,
        )

        step_result = aggregate_eval_results(step_eval_results)
        print(format_step_log(step, max_step, step_result))

        milestone_results.append({
            "step": step,
            "edit_time": edit_time,
            "total_edit_time": total_edit_time,
            "edits": [
                {
                    "prompt": req.prompt,
                    "old_ground_truth": req.old_ground_truth,
                    "new_ground_truth": req.new_ground_truth,
                }
                for req in edit_requests[:step]
            ],
            "result": asdict(step_result),
        })

    elapsed = perf_counter() - run_start
    avg_edit_time = total_edit_time / max_step
    print(
        "[method-end] "
        f"run={run_index}/{n_runs} "
        f"method={editing_method} "
        f"elapsed={elapsed:.2f}s ({elapsed / 60:.2f} min) "
        f"avg_edit_time={avg_edit_time:.2f}s"
    )
    return milestone_results


async def run_benchmark(
    dataset_name: str,
    dataset,
    build_pair: Callable[[object], tuple[EditRequest, EvalResult]],
) -> None:

    if not eval_milestones:
        raise ValueError("eval_milestones must not be empty")
    if eval_milestones[0] <= 0:
        raise ValueError("Milestones must be positive integers")

    global GLOBAL_MODEL, GLOBAL_TOKENIZER
    max_step = eval_milestones[-1]

    print(
        "[run-start] "
        f"methods={editing_methods} "
        f"model={model_name} "
        f"dataset={dataset_name}-{dataset_variant} "
        f"milestones={eval_milestones} "
        f"eval_batch_size={eval_batch_size}"
    )

    dataset_start = perf_counter()
    for run_index in range(1, n_runs + 1):
        run_seed = base_seed + run_index - 1
        edit_requests, eval_results = build_inputs(dataset, build_pair, run_seed)

        print(
            "[run-sample] "
            f"run={run_index}/{n_runs} "
            f"seed={run_seed}"
        )

        for editing_method in editing_methods:
            editor = load_editor(
                f"../src/config/{editing_method}/{model_name}.yaml",
                model=GLOBAL_MODEL,
                tokenizer=GLOBAL_TOKENIZER,
            )
            try:
                if GLOBAL_MODEL is None or GLOBAL_TOKENIZER is None:
                    GLOBAL_MODEL = editor.model
                    GLOBAL_TOKENIZER = editor.tokenizer

                milestone_results = await run_method_benchmark(
                    editor=editor,
                    edit_requests=edit_requests,
                    eval_results=eval_results,
                    editing_method=editing_method,
                    max_step=max_step,
                    run_index=run_index,
                )

                save_benchmark_results(
                    results=milestone_results,
                    dataset_variant=dataset_variant,
                    edited_model=model_name,
                    editing_method=editing_method,
                    milestones=eval_milestones,
                    dataset_name=dataset_name,
                    run_index=run_index,
                )
            finally:
                editor.reset()
                del editor

    elapsed = perf_counter() - dataset_start
    print(
        "[run-end] "
        f"methods={editing_methods} "
        f"model={model_name} "
        f"dataset={dataset_name}-{dataset_variant} "
        f"milestones={eval_milestones} "
        f"eval_batch_size={eval_batch_size} "
        f"elapsed={elapsed:.2f}s ({elapsed / 60:.2f} min)"
    )


In [ ]:
def build_cfact_pair(record) -> tuple[EditRequest, EvalResult]:
    edit_request = EditRequest(record["subject"], record["prompt"], record["old_ans"], record["new_ans"])
    eval_result = EvalResult(
        success=MetricResult(
            details=[EvalRecord(record["prompt"], record["new_ans"])] +
                    [EvalRecord(prompt, record["new_ans"]) for prompt in record["rephrases"]]
        ),
        locality=MetricResult(
            details=[EvalRecord(pair["prompt"], pair["ans"]) for pair in record["locality"]]
        ),
    )
    return edit_request, eval_result


await run_benchmark("cfact", cfact[dataset_variant], build_cfact_pair)


In [ ]:
def build_mquake_pair(record) -> tuple[EditRequest, EvalResult]:
    edit_request = EditRequest(record["subject"], record["prompt"], record["old_ans"], record["new_ans"])
    eval_result = EvalResult(
        portability=MetricResult(
            details=[
                EvalRecord(prompt, record["portability"]["new_ans"])
                for prompt in record["portability"]["prompts"]
            ]
        )
    )
    return edit_request, eval_result


await run_benchmark("mquake", mquake[dataset_variant], build_mquake_pair)
